In [1]:
import networkx as nx

from QHyper.problems.community_detection import CommunityDetectionProblem, Network
from QHyper.solvers.quantum_annealing.dwave.advantage import Advantage

import numpy as np
import pickle

%load_ext autoreload
%autoreload 2

In [2]:
G = nx.powerlaw_cluster_graph(10, 1, 0.01)
network = Network(graph=G)
problem = CommunityDetectionProblem(network_data=network, communities=2, one_hot_encoding=False)

In [3]:
adv = Advantage(problem=problem, num_reads=2, use_clique_embedding=True, elapse_times=True, chain_strength=0.0001)

In [4]:
id = 3
res = adv.solve(return_metadata=True, saving_path=f"id_{id}_sampleset_adv")

Could not save sampleset to .json file: 'SampleSet' object has no attribute 'to_file'


c:\Users\basia\Desktop\Praca_Inzynierska_2024\QHyper\QHyper\solvers\quantum_annealing\dwave\advantage.py:220: UserWarning: No timing information available for the sampleset. 
  warnings.warn(


In [12]:
with open(f"id_{id}_sampleset_adv_serializable.pkl", "rb") as f:
    l = pickle.load(f)

In [13]:
from dimod import SampleSet


ls = SampleSet.from_serializable(l)

In [14]:
sampleset = ls
print("Number of nodes in one set is {}, in the other, {}. \nEnergy is {}.".format(
    sum(sampleset.first.sample.values()),
    10 - sum(sampleset.first.sample.values()),
    sampleset.first.energy))     
print("Percentage of samples with >5 percent chain breaks is {}.".format(
    np.count_nonzero(sampleset.record.chain_break_fraction > 0.05)/2*100))  

Number of nodes in one set is 7, in the other, 3. 
Energy is -1.1111111111111105.
Percentage of samples with >5 percent chain breaks is 100.0.


In [ ]:
# import dimod


# dimod.as_samples(ls)

(array([[1, 0, 0, 0, 1, 0, 1, 1, 0, 0],
        [0, 1, 0, 1, 0, 0, 0, 1, 1, 0]], dtype=int8),
 ['x0', 'x1', 'x2', 'x3', 'x4', 'x5', 'x6', 'x7', 'x8', 'x9'])

In [15]:
def communities_to_list(sample, communities_number) -> list:
    communities = []
    for k in range(communities_number):
        subcommunity = []
        for i in sample:
            if sample[i] == k:
                key = int(i[1:])
                subcommunity.append(key)
        communities.append(subcommunity)

    return communities

In [16]:
communities_to_list(ls.first.sample, 2)

[[2, 3, 9], [0, 1, 4, 5, 6, 7, 8]]

In [24]:
nx.community.modularity(G, communities_to_list(ls.first.sample, 2))

0.12345679012345673

In [18]:
ls.info["embedding_context"]["chain_strength"]

0.0001

In [19]:
ls.record.chain_break_fraction

array([0.5, 0.4])

In [20]:
sampleset = ls
print("Percentage of samples with high rates of breaks is {}.".format(
       np.count_nonzero(sampleset.record.chain_break_fraction > 0.33)/2*100))    

Percentage of samples with high rates of breaks is 100.0.


In [21]:
problem_id = ls.info["problem_id"]
problem_id

'29722194-e555-4b1f-9f80-7b98a4c8fef3'

In [ ]:
from dwave.inspector import storage

# try:
#     pd = storage.get_problem(problem_id)
# except KeyError:
#     pd = None
# pd

In [ ]:
pd = storage.get_problem(problem_id)

In [ ]:
# from dwave.inspector.adapters import enable_data_capture

# enable_data_capture()

In [22]:
ls

SampleSet(rec.array([([1, 0, 1, 1, 1, 1, 0, 1, 1, 1], -0.61111111, 1, 0.5),
           ([1, 1, 0, 0, 1, 1, 1, 1, 1, 0], -1.11111111, 1, 0.4)],
          dtype=[('sample', 'i1', (10,)), ('energy', '<f8'), ('num_occurrences', '<i8'), ('chain_break_fraction', '<f8')]), Variables(['x0', 'x1', 'x2', 'x3', 'x4', 'x5', 'x6', 'x7', 'x8', 'x9']), {'timing': {'qpu_sampling_time': 253.2, 'qpu_anneal_time_per_sample': 20.0, 'qpu_readout_time_per_sample': 86.02, 'qpu_access_time': 16016.36, 'qpu_access_overhead_time': 742.64, 'qpu_programming_time': 15763.16, 'qpu_delay_time_per_sample': 20.58, 'total_post_processing_time': 132.0, 'post_processing_overhead_time': 132.0}, 'problem_id': '29722194-e555-4b1f-9f80-7b98a4c8fef3', 'embedding_context': {'embedding': {'x0': [180, 2940], 'x1': [195, 2955], 'x2': [150, 2970], 'x3': [165, 2985], 'x4': [210, 3000], 'x5': [225, 3015], 'x6': [240, 3030], 'x7': [255, 3045], 'x8': [120, 3060], 'x9': [135, 3075]}, 'chain_break_method': 'majority_vote', 'embedding_pa

In [23]:
import dwave.inspector
from dwave.inspector import show

show(ls)

Serving Inspector on http://127.0.0.1:18000/?problemId=29722194-e555-4b1f-9f80-7b98a4c8fef3

'http://127.0.0.1:18000/?problemId=29722194-e555-4b1f-9f80-7b98a4c8fef3'

In [31]:
from dwave.system import DWaveSampler
from dwave.system.composites import FixedEmbeddingComposite


version = "Advantage_system4.1"
region = "na-west-1"

sampler = DWaveSampler(solver=version, region=region)
fixed_embedding_composite = FixedEmbeddingComposite(sampler, embedding=ls.info["embedding_context"]["embedding"])

In [74]:
fixed_embedding_composite.warnings_default

<WarningAction.SAVE: 'save'>

In [75]:
sampleset.info["warnings"]

[{'type': dwave.system.warnings.ChainStrengthWarning,
  'message': 'Some quadratic biases are stronger than the given chain strength',
  'level': 30,
  'data': {'source_interactions': [['x1', 'x0'],
    ['x2', 'x0'],
    ['x2', 'x1'],
    ['x3', 'x0'],
    ['x3', 'x1'],
    ['x3', 'x2'],
    ['x4', 'x0'],
    ['x4', 'x1'],
    ['x4', 'x2'],
    ['x4', 'x3'],
    ['x5', 'x0'],
    ['x5', 'x1'],
    ['x5', 'x2'],
    ['x5', 'x3'],
    ['x5', 'x4'],
    ['x6', 'x0'],
    ['x6', 'x1'],
    ['x6', 'x2'],
    ['x6', 'x3'],
    ['x6', 'x4'],
    ['x6', 'x5'],
    ['x7', 'x0'],
    ['x7', 'x1'],
    ['x7', 'x2'],
    ['x7', 'x3'],
    ['x7', 'x4'],
    ['x7', 'x5'],
    ['x7', 'x6'],
    ['x8', 'x0'],
    ['x8', 'x1'],
    ['x8', 'x2'],
    ['x8', 'x3'],
    ['x8', 'x4'],
    ['x8', 'x5'],
    ['x8', 'x6'],
    ['x8', 'x7'],
    ['x9', 'x0'],
    ['x9', 'x1'],
    ['x9', 'x2'],
    ['x9', 'x3'],
    ['x9', 'x4'],
    ['x9', 'x5'],
    ['x9', 'x6'],
    ['x9', 'x7'],
    ['x9', 'x8']]}},
 {'typ

In [78]:
dwave.system.warnings.ChainBreakWarning

dwave.system.warnings.ChainBreakWarning

In [94]:
G.number_of_nodes()

20